In [50]:
import pandas as pd
import json
import glob

In [51]:
results_dir = '../results'

In [52]:
all_results = glob.glob('../results/*.json')

table_data = []
for result in all_results:
    with open(result, 'r') as f:
        data = json.load(f)
        for k, v in data.items():
            if k in ['Accuracy', 'IoU', 'AP']:
                data[k] = v['mean']        
            if k == 'train_config':
                data[k] = str(v)
        table_data.append(data)

In [53]:
df = pd.DataFrame(table_data)
df

,AP,IoU,Accuracy,number_of_samples,depth_channel_representation_mode,window_len,train_config,multiple_digits,model_path
0,0.133868,0.111126,0.889732,1184,window_time_normalized,50,"{'epochs': 2, 'learning_rate': 0.001, 'layers'...",False,C:\Users\Emiel\Documents\TU\dsait4205-ev-mask-...
1,0.222128,0.209604,0.926805,1184,window_time_normalized,50,"{'epochs': 2, 'learning_rate': 0.001, 'layers'...",False,C:\Users\Emiel\Documents\TU\dsait4205-ev-mask-...
2,0.192568,0.199624,0.928853,1184,window_time_normalized,50,"{'epochs': 2, 'learning_rate': 0.001, 'layers'...",False,C:\Users\Emiel\Documents\TU\dsait4205-ev-mask-...


In [55]:
metrics = ['Accuracy', 'IoU', 'AP']
grouped = df.groupby(['window_len','train_config','multiple_digits','depth_channel_representation_mode'])
means = grouped[metrics].mean(numeric_only=True)
stds = grouped[metrics].std(numeric_only=True)
stds.rename(columns=lambda x: x + '_std', inplace=True)
counts = pd.DataFrame(grouped.size(), columns=['n'])

combined = pd.concat([means, stds, counts], axis=1)
for metric in metrics:
    combined[metric] = combined.apply(lambda x: f"{x[metric]:.2f} ± {x[f'{metric}_std']:.2f}", axis=1)
    combined.drop([f'{metric}_std'], axis=1, inplace=True)
combined.to_csv('results.csv', index=True)
combined

,,,,Accuracy,IoU,AP,n
window_len,train_config,multiple_digits,depth_channel_representation_mode,,,,
50,"{'epochs': 2, 'learning_rate': 0.001, 'layers': 'heads'}",False,window_time_normalized,0.92 ± 0.02,0.17 ± 0.05,0.18 ± 0.04,3
